# Import Libraries

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

# Data preparation
data = {
    'Dataset Size': [1000, 5000, 10000, 15000],
    'Levenshtein_Standard': [219.24, 223.42, 223.16, 225.68],
    'Levenshtein_Improved': [224.36, None, 219.74, 220.42],  # None for missing 5000 improved
    'ROUGE1_F1_Standard': [0.1012, 0.1066, 0.1007, 0.1016],
    'ROUGE1_F1_Improved': [0.1063, None, 0.1032, 0.1046],
    'ROUGE1_Precision_Standard': [0.0850, 0.0888, 0.0825, 0.0840],
    'ROUGE1_Precision_Improved': [0.0882, None, 0.0857, 0.0890],
    'ROUGE1_Recall_Standard': [0.1297, 0.1381, 0.1337, 0.1332],
    'ROUGE1_Recall_Improved': [0.1375, None, 0.1344, 0.1329],
    'ROUGE2_F1_Standard': [0.0315, 0.0304, 0.0298, 0.0297],
    'ROUGE2_F1_Improved': [0.0305, None, 0.0299, 0.0320],
    'ROUGE2_Precision_Standard': [0.0262, 0.0247, 0.0237, 0.0237],
    'ROUGE2_Precision_Improved': [0.0246, None, 0.0240, 0.0267],
    'ROUGE2_Recall_Standard': [0.0416, 0.0416, 0.0416, 0.0416],
    'ROUGE2_Recall_Improved': [0.0416, None, 0.0416, 0.0416],
    'ROUGEL_F1_Standard': [0.0909, 0.0951, 0.0902, 0.0876],
    'ROUGEL_F1_Improved': [0.0957, None, 0.0924, 0.0943],
    'ROUGEL_Precision_Standard': [0.0759, 0.0789, 0.0733, 0.0720],
    'ROUGEL_Precision_Improved': [0.0789, None, 0.0760, 0.0797],
    'ROUGEL_Recall_Standard': [0.1177, 0.1243, 0.1211, 0.1161],
    'ROUGEL_Recall_Improved': [0.1249, None, 0.1218, 0.1209]
}

df = pd.DataFrame(data)

# Plotting function
def plot_metric(metric, title, ylabel, filename):
    plt.figure(figsize=(10, 6))
    plt.plot(df['Dataset Size'], df[f'{metric}_Standard'], marker='o', label='Standard', color='blue')
    plt.plot(df['Dataset Size'], df[f'{metric}_Improved'], marker='s', label='Improved', color='orange')
    plt.title(title)
    plt.xlabel('Dataset Size')
    plt.ylabel(ylabel)
    plt.xticks(df['Dataset Size'])
    plt.legend()
    plt.grid(True)
    plt.savefig(filename)
    plt.close()

# Plot Levenshtein Distance
plot_metric('Levenshtein', 'Average Levenshtein Distance vs Dataset Size', 'Levenshtein Distance', 'levenshtein_distance.png')

# Plot ROUGE-1 Scores
plot_metric('ROUGE1_F1', 'ROUGE-1 F1 Score vs Dataset Size', 'F1 Score', 'rouge1_f1.png')
plot_metric('ROUGE1_Precision', 'ROUGE-1 Precision vs Dataset Size', 'Precision', 'rouge1_precision.png')
plot_metric('ROUGE1_Recall', 'ROUGE-1 Recall vs Dataset Size', 'Recall', 'rouge1_recall.png')

# Plot ROUGE-2 Scores
plot_metric('ROUGE2_F1', 'ROUGE-2 F1 Score vs Dataset Size', 'F1 Score', 'rouge2_f1.png')
plot_metric('ROUGE2_Precision', 'ROUGE-2 Precision vs Dataset Size', 'Precision', 'rouge2_precision.png')
plot_metric('ROUGE2_Recall', 'ROUGE-2 Recall vs Dataset Size', 'Recall', 'rouge2_recall.png')

# Plot ROUGE-L Scores
plot_metric('ROUGEL_F1', 'ROUGE-L F1 Score vs Dataset Size', 'F1 Score', 'rougel_f1.png')
plot_metric('ROUGEL_Precision', 'ROUGE-L Precision vs Dataset Size', 'Precision', 'rougel_precision.png')
plot_metric('ROUGEL_Recall', 'ROUGE-L Recall vs Dataset Size', 'Recall', 'rougel_recall.png')

print("Plots saved as PNG files.")

Plots saved as PNG files.


In [3]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from model import GPTLanguageModel
import argparse
from rouge import Rouge
from utils import *
import string

# Load the Parsing Parameter Utilities

In [5]:
def parse_option():
    parser = argparse.ArgumentParser('argument for training')

    parser.add_argument('--batch_size', type=int, default=256,
                        help='batch_size')
    parser.add_argument('--block_size', type=int, default=256,
                        help='Size of blocks to process vocabulary')

    parser.add_argument('--max_iters', type=int, default=256,
                        help='Max Iterations of the Training Process')

    parser.add_argument('--eval_interval', type=int, default=256,
                        help='Max Iterations of the Training Process')


    # optimization
    parser.add_argument('--learning_rate', type=float, default=3e-4,
                        help='learning rate')
    parser.add_argument('--dropout', type=float, default=0.2,
                        help='dropout')

    parser.add_argument('--momentum', type=float, default=0.9,
                        help='momentum')

    # model dataset
    parser.add_argument('--model', type=str, default='basic')
    parser.add_argument('--save_file', type=str, default='./models/test.pth')
    parser.add_argument('--ckpt', type=str, default='')
    parser.add_argument('--optimizer', type=str, choices=['SGD', 'Adam'], default='SGD')
    parser.add_argument('--n_heads', type=int, default=6, help='Number of Heads in Attention Block')
    parser.add_argument('--n_layer', type=int, default=6, help='Number of Layers in Attention Block')
    parser.add_argument('--n_embd', type=int, default=384, help='Embedding dimension')
    parser.add_argument('--loss', type=str, default='NLL')
    parser.add_argument('--training_file', type=str, default='./train_data/shakespeare.txt')
    parser.add_argument('--device', type=str, default='cuda:0')
    parser.add_argument('--dataset', type=str, default='shakespeare',choices=['shakespeare'], help='dataset')
    parser.add_argument('')



    opt = parser.parse_args([])


    return opt
opt = parse_option()

usage: argument for training [-h] [--batch_size BATCH_SIZE]
                             [--block_size BLOCK_SIZE]
                             [--vocab_size VOCAB_SIZE] [--dropout DROPOUT]
                             [--model MODEL] [--ckpt CKPT] [--n_heads N_HEADS]
                             [--n_layer N_LAYER] [--n_embd N_EMBD]
                             [--loss LOSS]
                             [--testing_file_prompt TESTING_FILE_PROMPT]
                             [--testing_file_response TESTING_FILE_RESPONSE]
                             [--testing_file_answer TESTING_FILE_ANSWER]
                             [--training_file TRAINING_FILE]
                             [--generate_token_number GENERATE_TOKEN_NUMBER]
                             [--device DEVICE] [--dataset {shakespeare}]
argument for training: error: unrecognized arguments: -f /home/hice1/zzhang3235/.local/share/jupyter/runtime/kernel-1cc8a4bf-bac5-4539-90e9-0a4c62c6dde4.json


SystemExit: 2

/usr/local/pace-apps/manual/packages/anaconda3/2023.03/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3513: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Load the Dataset

In [ ]:
#### Load a Text File with the Data of Interest ####
with open('./train_data/shakespeare.txt','r',encoding='utf-8') as f:
    text = f.read()

#### Get the Length of the text ####
print(len(text))

#### Get a List of all Potential Characters ####
ascii = string.printable


# Convert the string to a list
chars = list(ascii)

vocab_size = len(chars)

#### create a mapping from characters to integers ####
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}


#### Createa function that will convert between integer character encodings and the original characters ####
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

#### Create a Training and Validation Set ####

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# Load the GPT Model

In [3]:
#### Initialize the Language Model ####
# Vocab Size: Number of Different Characters that are possible within the language corpus of interest (Upper Case + Lower Case + Numbers + Punctuation)
# n_embd: Size of the vectors used to represent the text within the model (Generated with Neural Networks)
# block_size: Number of characters to use as context when predicting the next probable character
# dropout: number of weights to drop from specific layer (used to help generalization of model)
# device: The gpu that the model will be loaded onto.

model = GPTLanguageModel(vocab_size, n_embd=384, block_size=256, dropout=0.2, device= 'cuda:0')
model = model.to('cuda:0')

NameError: name 'vocab_size' is not defined

# Perform Model Training

In [9]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for iter in range(opt.max_iters):

    # sample a batch of data
    xb, yb = get_batch('train',train_data,val_data,opt)

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Load Prompt and the Associated Answer to test the Model

In [10]:
with open(opt.testing_file_prompt, 'r', encoding='utf-8') as f:
    text_test_prompt = f.read()

with open(opt.testing_file_answer, 'r', encoding='utf-8') as f:
    text_test_answer= f.read()

#### Encode the Prompt into a numerical form that can be input into your model ####
context = torch.tensor(encode(text_test_prompt), device='cuda:0').unsqueeze(dim=-1)

AttributeError: 'Namespace' object has no attribute 'testing_file_prompt'

# Generate the Response from your LLM

In [ ]:
#### You want your model to generate the same number of characters as values in your response ####


#### Note that the testing files will not contain any characters that are not in the vocabulary of the LLM! ####
### This is a common problem with LLM systems. How can we overcome it? ####
number_gen = len(text_test_answer)
response = decode(model.generate(context, max_new_tokens=number_gen,block_size=opt.block_size)[0].tolist())

# Compute a Metric between the True Answer and your Response

In [ ]:
#### We use the Rouge Metric in this Example ####
rouge = Rouge()
scores = rouge.get_scores(response, text_test_answer)
print(scores)